# 03 — Modélisation & Évaluation

Ce notebook entraîne les 3 modèles de classification supervisée et analyse leurs performances.

**Prérequis :** avoir lancé `scripts/train.py` pour générer `data/processed/allego_labeled.csv` et les modèles dans `models/`.

In [ ]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
PLOTS_DIR  = ROOT / "plots"
MODELS_DIR = ROOT / "models"

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120

FEATURE_COLS = [
    "puissance_nominale", "prise_type_ef", "prise_type_2",
    "prise_type_combo_ccs", "prise_type_chademo", "prise_type_autre",
    "implantation_encoded", "acces_libre", "nbre_pdc", "latitude", "longitude",
]
LABEL_NAMES  = {0: "Sous-équipé", 1: "Normal", 2: "Bien équipé"}
CLASS_NAMES  = ["Sous-équipé", "Normal", "Bien équipé"]


## 1. Chargement du dataset traité

In [ ]:
df = pd.read_csv(ROOT / "data" / "processed" / "allego_labeled.csv")
X  = df[FEATURE_COLS]
y  = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset : {len(df):,} PDC | {len(FEATURE_COLS)} features | 3 classes")
print(f"Train   : {len(X_train):,} | Test : {len(X_test):,}")
print()
print("Répartition des classes :")
print(y.map(LABEL_NAMES).value_counts())


## 2. Chargement des modèles entraînés

In [ ]:
models = {
    "Logistic Regression": joblib.load(MODELS_DIR / "logistic_regression.joblib"),
    "K-Nearest Neighbors": joblib.load(MODELS_DIR / "knn.joblib"),
    "XGBoost"            : joblib.load(MODELS_DIR / "xgboost.joblib"),
}
print("Modèles chargés :", list(models.keys()))


## 3. Évaluation sur le jeu de test

In [ ]:
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    results[name] = {
        "accuracy"   : accuracy_score(y_test, y_pred),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "f1_macro"   : f1_score(y_test, y_pred, average="macro"),
        "y_pred"     : y_pred,
    }
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 4. Comparaison des métriques

In [ ]:
metrics_df = pd.DataFrame([
    {"Modèle": name, "Accuracy": v["accuracy"],
     "F1 weighted": v["f1_weighted"], "F1 macro": v["f1_macro"]}
    for name, v in results.items()
])
print(metrics_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(metrics_df))
width = 0.25
colors = ["#3498db", "#e74c3c", "#2ecc71"]
metrics_to_plot = ["Accuracy", "F1 weighted", "F1 macro"]

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, metrics_df[metric], width, label=metric, color=colors[i], alpha=0.85)
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_df["Modèle"], fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Comparaison des modèles — Jeu de test Allego")
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "09_comparaison_modeles.png")
plt.show()


## 5. Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, v) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, v["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=11)
    ax.tick_params(axis="x", labelrotation=30)

plt.suptitle("Matrices de confusion — Jeu de test", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "10_matrices_confusion.png", bbox_inches="tight")
plt.show()


## 6. Importance des features (XGBoost)

In [ ]:
xgb_model = models["XGBoost"]
importances = xgb_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ["#e74c3c" if f in ["latitude", "longitude"] else "#3498db" for f in feat_imp.index]
feat_imp.plot(kind="barh", ax=ax, color=colors_bar)
ax.set_title("Importance des features — XGBoost")
ax.set_xlabel("Importance (gain)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "11_feature_importance_xgboost.png")
plt.show()


## 7. Conclusion

| Modèle | Accuracy | F1 weighted | Commentaire |
|---|---|---|---|
| Logistic Regression | ~60% | ~59% | Baseline faible — problème non linéaire |
| KNN | ~74% | ~74% | Amélioration notable grâce à la distance |
| **XGBoost** | **~99%** | **~99%** | Meilleur modèle — capture les patterns géographiques |

XGBoost domine grâce à sa capacité à modéliser les interactions non linéaires entre les features géographiques (latitude/longitude) et les caractéristiques des bornes.